# Controller safety under bounded disturbances

Train a small controller, inspect its losses and distinguish an invariant certificate from sampled rollouts. Actions are bounded by ±0.1. The next state must stay inside [-1, 1]² and outside the central obstacle (-0.3, 0.3)² for every disturbance in ±0.01².

This notebook defaults to a **two-epoch CPU execution check** with generated teacher data. It does not report a benchmark score. Start from a repository checkout using the [case-study setup](index.md).

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import tempfile

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "case_studies").is_dir() and (p / "pyproject.toml").is_file())
sys.path.insert(0, str(REPO))
WORK = Path(tempfile.mkdtemp(prefix="ctrain-case-study-"))
print("Fresh outputs:", WORK)


## Run through the library wrapper

The entry point constructs a library model and property wrapper, then calls `train_model`. It also records checkpoint and source provenance.

In [ ]:
output = WORK / "controller"
command = [sys.executable, "-m", "case_studies.controller.run",
           "--smoke", "--device", "cpu", "--output", str(output)]
completed = subprocess.run(command, cwd=REPO, check=True, capture_output=True, text=True)
result = json.loads((output / "results.json").read_text())
print(json.dumps(result["metrics"], indent=2))

## Inspect the geometry and training signal

The shaded square is excluded from the safe region. A low task loss or safe sampled rollout alone does not certify invariance.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from case_studies.controller.run import state_boxes

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for box in state_boxes():
    lower, upper = box.lower, box.upper
    axes[0].add_patch(Rectangle(lower.tolist(), float(upper[0]-lower[0]),
                               float(upper[1]-lower[1]), fill=False, edgecolor="#3977ad"))
axes[0].add_patch(Rectangle((-.3, -.3), .6, .6, color="#d87965", alpha=.6))
axes[0].set(xlim=(-1.1, 1.1), ylim=(-1.1, 1.1), aspect="equal", title="Declared state regions")
history = result["history"]
for key in ("task_loss", "property_loss"):
    axes[1].plot([r["epoch"] for r in history], [r[key] for r in history], label=key)
axes[1].set(xlabel="Epoch", ylabel="Loss", title="CPU execution check")
axes[1].legend()
fig.tight_layout()
plt.show()

## Full experiment

From the repository root:

```bash
python -m case_studies.controller.run --teacher safety --epochs 100 --device cpu --output runs/case_studies/controller_safety
```

Use a fresh output path, `--method` / `--training-options` for another strategy and `--complete` for optional complete verification. Unknown results remain unresolved. Goal reaching is empirical and is not part of the invariant specification. [Full protocol](https://github.com/ADA-research/CTRAIN/blob/dev/case_studies/controller/README.md).